# LatentDriver Validation Preprocess (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_preprocess_val_colab.ipynb)

This notebook prepares either a **smoke validation subset** or the **full validation preprocessing artifacts** required for Waymax evaluation, using either a **Drive-local WOMD root** or an **authenticated WOMD GCS root**.


## What this notebook does

1. sync this repo and install the local package
2. mount Google Drive and bind the canonical asset layout
3. clone and patch the pinned LatentDriver fork
4. optionally install the heavy Waymax/JAX/Torch runtime
5. choose whether WOMD validation comes from Drive or authenticated GCS
6. for smoke mode, stage one validation shard locally if needed
7. prepare a one-shard smoke TFRecord if needed
8. run the upstream validation preprocessing job


In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


$ git clone --branch main https://github.com/achyutmorang/latentdriver-waymax-experiments.git /content/latentdriver-waymax-experiments
Cloning into '/content/latentdriver-waymax-experiments'...
$ /usr/bin/python3 -m pip install -e /content/latentdriver-waymax-experiments
Obtaining file:///content/latentdriver-waymax-experiments
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for latentdriver-waymax-experiments (pyproject.toml): started
  Building editable for latentdriver-waymax-experiments (pyproject.toml): finished with stat

In [2]:

from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


Mounted at /content/drive
{
  "checkpoints": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/checkpoints",
  "drive_root": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments",
  "preprocessed": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/preprocessed",
  "results": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/results/runs",
  "smoke": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/smoke"
}


In [3]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


$ /usr/bin/python3 scripts/bootstrap_upstream.py
Cloning into '/content/latentdriver-waymax-experiments/external/LatentDriver'...
Note: switching to '721bd6e1f87373457b743d0e0a9606d4d0727b6f'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 721bd6e Update README.md
{
  "fork_repo_url": "https://github.com/achyutmorang/LatentDriver.git",
  "patch_path": "/content/latentdriver-waymax-experiments/patches/latentdriver_eval_contract.patch",
  "patch_state": "applied",
  "pinned_commit": "721bd6e1f87373457b743d0e0a96

In [4]:
INSTALL_RUNTIME = True

if INSTALL_RUNTIME:
    run([sys.executable, "scripts/setup_colab_runtime.py", "--editable-project"], cwd=REPO_DIR)
else:
    print("Skipping runtime installation. Set INSTALL_RUNTIME=True if this is a fresh Colab session.")


$ /usr/bin/python3 scripts/setup_colab_runtime.py --editable-project
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.7 MB/s eta 0:00:00
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Cloning https://github.com/waymo-research/waymax.git (to revision main) to /tmp/pip-install-qk9lx_cd/waymo-waymax_953c893cc60347f3b93953e701002666
  Running command git clone --f

In [5]:
from __future__ import annotations

import json
from pathlib import Path

WOMD_SOURCE_MODE = 'gcs'  # 'drive' or 'gcs'
WAYMO_DATASET_DRIVE_ROOT = '/content/drive/MyDrive/waymo_open_dataset'
WOMD_GCS_DATASET_ROOT = 'gs://waymo_open_dataset_motion_v_1_1_0'
PREPROCESS_MODE = 'full'  # switch to 'full' after smoke passes
SMOKE_SHARD_INDEX = 0
GCS_STAGING_ROOT = str(Path(binding["drive_root"]) / "assets" / "raw_womd")

print(json.dumps({
    'WOMD_SOURCE_MODE': WOMD_SOURCE_MODE,
    'WAYMO_DATASET_DRIVE_ROOT': WAYMO_DATASET_DRIVE_ROOT,
    'WOMD_GCS_DATASET_ROOT': WOMD_GCS_DATASET_ROOT,
    'PREPROCESS_MODE': PREPROCESS_MODE,
    'SMOKE_SHARD_INDEX': SMOKE_SHARD_INDEX,
    'GCS_STAGING_ROOT': GCS_STAGING_ROOT,
}, indent=2, sort_keys=True))


{
  "GCS_STAGING_ROOT": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/raw_womd",
  "PREPROCESS_MODE": "full",
  "SMOKE_SHARD_INDEX": 0,
  "WAYMO_DATASET_DRIVE_ROOT": "/content/drive/MyDrive/waymo_open_dataset",
  "WOMD_GCS_DATASET_ROOT": "gs://waymo_open_dataset_motion_v_1_1_0",
  "WOMD_SOURCE_MODE": "gcs"
}


In [6]:
from __future__ import annotations

import json
import os
import subprocess
import sys

sys.path.insert(0, str(REPO_DIR / 'src'))

from latentdriver_waymax_experiments.womd import probe_gcs_uri

if WOMD_SOURCE_MODE not in {'drive', 'gcs'}:
    raise ValueError(f'Unsupported WOMD_SOURCE_MODE={WOMD_SOURCE_MODE!r}')

resolved_dataset_root = WAYMO_DATASET_DRIVE_ROOT
staging_summary = None
gcs_probe_cli = None
gcs_probe_command = None
gcs_probe_lines: list[str] = []

if WOMD_SOURCE_MODE == 'gcs':
    from google.colab import auth

    auth.authenticate_user()
    adc = subprocess.run(
        ['gcloud', 'auth', 'application-default', 'print-access-token'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    if adc.returncode != 0:
        raise RuntimeError('Failed to acquire application-default credentials for WOMD GCS access.')

    validation_probe = WOMD_GCS_DATASET_ROOT.rstrip('/') + '/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150'
    probe = probe_gcs_uri(validation_probe)
    gcs_probe_cli = probe['cli']
    gcs_probe_command = probe['command']
    gcs_probe_lines = probe['stdout_lines']

    if PREPROCESS_MODE == 'smoke':
        staging_summary = json.loads(run([
            sys.executable,
            'scripts/stage_womd_validation_shard.py',
            '--gcs-root', WOMD_GCS_DATASET_ROOT,
            '--staging-root', GCS_STAGING_ROOT,
            '--shard-index', str(SMOKE_SHARD_INDEX),
        ], cwd=REPO_DIR, stream=False))
        resolved_dataset_root = staging_summary['staging_root']
    else:
        resolved_dataset_root = WOMD_GCS_DATASET_ROOT

os.environ['LATENTDRIVER_WAYMO_DATASET_ROOT'] = resolved_dataset_root
print(json.dumps({
    'LATENTDRIVER_WAYMO_DATASET_ROOT': os.environ['LATENTDRIVER_WAYMO_DATASET_ROOT'],
    'PREPROCESS_MODE': PREPROCESS_MODE,
    'SMOKE_SHARD_INDEX': SMOKE_SHARD_INDEX,
    'WOMD_SOURCE_MODE': WOMD_SOURCE_MODE,
    'gcs_probe_cli': gcs_probe_cli,
    'gcs_probe_command': gcs_probe_command,
    'gcs_probe_matches': gcs_probe_lines,
    'staging_summary': staging_summary,
}, indent=2, sort_keys=True))


{
  "LATENTDRIVER_WAYMO_DATASET_ROOT": "gs://waymo_open_dataset_motion_v_1_1_0",
  "PREPROCESS_MODE": "full",
  "SMOKE_SHARD_INDEX": 0,
  "WOMD_SOURCE_MODE": "gcs",
  "gcs_probe_cli": "gcloud-storage",
  "gcs_probe_command": [
    "gcloud",
    "storage",
    "ls",
    "gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150"
  ],
  "gcs_probe_matches": [
    "gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150"
  ],
  "staging_summary": null
}


In [7]:
if PREPROCESS_MODE == 'smoke':
    run([sys.executable, "scripts/prepare_smoke_subset.py", "--shard-index", str(SMOKE_SHARD_INDEX)], cwd=REPO_DIR)
else:
    print("Skipping smoke subset creation because PREPROCESS_MODE=full")

run([sys.executable, "scripts/preprocess_validation_only.py", "--mode", PREPROCESS_MODE, "--dry-run"], cwd=REPO_DIR)


Skipping smoke subset creation because PREPROCESS_MODE=full
$ /usr/bin/python3 scripts/preprocess_validation_only.py --mode full --dry-run
{
  "auto_force_partial": false,
  "cache_status": {
    "any_present": false,
    "complete": false,
    "intention_ready": false,
    "map_ready": false,
    "partial": false,
    "paths": {
      "intention_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_intention_label",
      "map_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path/map",
      "preprocess_root": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path",
      "route_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path/route"
    },
    "route_ready": false
  },
  "command": [
    "/usr/bin/python3",
    "-m",
    "src.preprocess.preprocess_data",
    "++batch_dims=[1,1000]",
    "++waymax_conf.p

'{\n  "auto_force_partial": false,\n  "cache_status": {\n    "any_present": false,\n    "complete": false,\n    "intention_ready": false,\n    "map_ready": false,\n    "partial": false,\n    "paths": {\n      "intention_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_intention_label",\n      "map_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path/map",\n      "preprocess_root": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path",\n      "route_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full/val_preprocessed_path/route"\n    },\n    "route_ready": false\n  },\n  "command": [\n    "/usr/bin/python3",\n    "-m",\n    "src.preprocess.preprocess_data",\n    "++batch_dims=[1,1000]",\n    "++waymax_conf.path=gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord@150",\n  

In [8]:
RUN_PREPROCESS = True

if RUN_PREPROCESS:
    cmd = [sys.executable, "scripts/preprocess_validation_only.py", "--mode", PREPROCESS_MODE, "--auto-force-partial"]
    run(cmd, cwd=REPO_DIR)
else:
    print("Skipping preprocessing run.")


$ /usr/bin/python3 scripts/preprocess_validation_only.py --mode full --auto-force-partial
E0000 00:00:1775868648.060002    4206 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775868648.071440    4206 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775868648.097060    4206 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775868648.097150    4206 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775868648.097178    4206 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775868648.097197   

RuntimeError: Command failed with exit code 247: /usr/bin/python3 scripts/preprocess_validation_only.py --mode full --auto-force-partial

In [ ]:
import json

config = json.loads((REPO_DIR / "configs" / "baselines" / "latentdriver_waymax_eval.json").read_text())
root = REPO_DIR / config["assets"]["preprocessed_root"] / PREPROCESS_MODE
summary = {}
for child in sorted(root.glob("*")):
    summary[child.name] = {
        "path": str(child),
        "exists": child.exists(),
        "entries": len(list(child.glob("*"))) if child.is_dir() else None,
    }
print(json.dumps(summary, indent=2, sort_keys=True))


In [ ]:
!readlink -f /content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/full


## Next step

Once preprocessing artifacts exist, run [`latentdriver_public_eval_colab.ipynb`](./latentdriver_public_eval_colab.ipynb) for checkpoint evaluation under a standardized Waymax contract.
